# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

**Unit of analysis (grain):** one row = one `content_hash_id`, for one `client_hash_id`,
on one `report_date`, a single content item's search performance on a single day.
This is the native grain of `fact_content_daily_performance`.

**Table(s) used:** `fact_content_daily_performance` (the daily fact, partitioned by
`month=YYYY-MM`) as the primary source. `dim_content` is used read-only, joined on
`content_hash_id`, purely for `position_tier` context and never aggregated.

**Time window:** development month is `2026-03` (mid-panel). The final month
(`2026-06`, exposed separately as `fact_content_daily_performance_sample`) is a
**sealed test month**  used later only to check query mechanics, never to shape
label logic, per the warning on the assignment card.

**What I'd predict/rank (label or proxy):** same proxy as weeks 1–2, `ctr_gap`,
the page's tier-average CTR minus its own observed CTR over a trailing window,
computed from `gsc_clicks` / `gsc_impressions` aggregated up from the daily grain.
Positive `ctr_gap` = page is under-monetizing its ranking position relative to peers
at the same `position_tier`. This is a proxy for "title/snippet worth reviewing,"
not a ground-truth label.

**One thing deliberately excluded:** `ga4_sessions` / any GA4 engagement column,
for rows where `ga4_data_available` is not `TRUE`. Early history for many clients
predates their GA4 rollout, and the flag can also be `NULL`, in both non-`TRUE`
cases the GA4 columns are either zero-filled or genuinely missing, not "zero
engagement." Since my lane's decision (CTR review) doesn't need GA4 at all, I
exclude those columns entirely rather than risk reading a zero-fill as a real zero.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
SAMPLE = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"

grain_probe = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {SAMPLE}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Grain probe on sample table, should be EMPTY, 0 rows = grain confirmed")
print(grain_probe)
print(f"Rows returned: {len(grain_probe)}")

Grain probe on sample table, should be EMPTY, 0 rows = grain confirmed
            client_hash_id           content_hash_id report_date  c
0  client_a2eeb8899886adde  content_5346109dab3709dd  2026-06-18  2
1  client_def0955f7a377868  content_7353d795fef06d72  2026-06-16  2
2  client_1a8bf67cad4ee525  content_8abad04c50e798f7  2026-06-25  2
3  client_e00b29e582949543  content_e3491394a9f3e2b3  2026-06-14  2
4  client_a22068e339bf95f5  content_11cf497afdf0034e  2026-06-17  2
Rows returned: 5


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features** (knowable at the decision moment, safe to use):
- `gsc_impressions` — daily search impressions for the content item
- `gsc_avg_position` — daily average ranking position
- `position_tier` (from `dim_content`, joined) — pre-computed position bucket
- `gsc_clicks` (trailing window, *not* the current day being scored) — used only to
  build the tier-average CTR baseline from *other* days, never the target day itself
- `gsc_data_available` — three-valued flag; filtered on, not modeled

**Label / proxy:** `ctr_gap = tier_avg_ctr - ctr`, where `ctr = gsc_clicks / gsc_impressions`
aggregated over the scoring window. This is what the whole lane predicts/ranks by —
it must never leak into the feature set as raw `gsc_clicks` from the *same* window
being scored.

**Context** (for joining/filtering, never modeled):
`client_hash_id`, `content_hash_id`, `report_date` — pseudonymous IDs and dates
used to group, join, and window, never learned from directly.

**Excluded:**
- `ga4_sessions`, `ga4_engaged_sessions`, any GA4 column — **why:** `ga4_data_available`
  is FALSE or NULL for a large share of early-history rows; using GA4 columns without
  strict `IS TRUE` filtering would silently read a zero-fill as "no engagement," and my
  lane's decision doesn't require GA4 at all.
- `sessions_ai` / AI-referral columns — **why:** extremely sparse in this slice (30,177
  of 78.8M rows warehouse-wide per the docs) — too thin to support a CTR-tier comparison
  without introducing noise dressed up as signal.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MARCH}
""").df()

print("Slice size and date span, March 2026")
print(slice_check)

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_null_flag_rows,
        COUNT(*) FILTER (WHERE gsc_impressions > 0 AND gsc_data_available IS TRUE) AS rows_with_real_impressions
    FROM {MARCH}
""").df()

print("Availability check, IS TRUE filter")
print(availability_check)
survive_pct = availability_check["gsc_available_rows"][0] / availability_check["total_rows"][0] * 100
print(f"{survive_pct:.1f} percent of March rows survive the IS TRUE filter")

Slice size and date span, March 2026
   total_rows  n_clients  n_content_items   min_date   max_date
0     9841378         55           331437 2026-03-01 2026-03-31
Availability check, IS TRUE filter
   total_rows  gsc_available_rows  gsc_null_flag_rows  \
0     9841378             3611061                   0   

   rows_with_real_impressions  
0                     3611061  
36.7 percent of March rows survive the IS TRUE filter


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1. **`gsc_impressions_prev` (trailing daily impressions, excluding scoring day)** —
   knowable at the decision moment because it's logged search exposure from *before*
   the day being scored, not the day itself.
2. **`gsc_avg_position_prev`** — knowable at the decision moment because ranking
   position on prior days is already observed and recorded by the time we score.
3. **`position_tier`** — knowable at the decision moment because it's a static
   bucket derived from historical average position, available in `dim_content`
   before any single day is scored.
4. **`ctr_prev` (clicks/impressions from the trailing window, excluding scoring day)**
   — knowable at the decision moment because it summarizes *past* click behavior,
   used only to build the peer-tier baseline, never the target day's own clicks.
5. **`has_min_volume` (`gsc_impressions_prev >= 100`)** — knowable at the decision
   moment because it's a simple threshold on a feature that's already knowable
   (item 1); it exists purely to filter out CTR estimates too noisy to trust.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

prev = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_prev,
        SUM(gsc_clicks) AS gsc_clicks_prev,
        AVG(gsc_avg_position) AS gsc_avg_position_prev
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date < DATE '2026-03-16'
    GROUP BY client_hash_id, content_hash_id
""").df()

target_day = con.sql(f"""
    SELECT client_hash_id, content_hash_id, gsc_clicks, gsc_impressions
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date = DATE '2026-03-31'
""").df()

prev["ctr_prev"] = prev["gsc_clicks_prev"] / prev["gsc_impressions_prev"].replace(0, np.nan)
prev["has_min_volume"] = (prev["gsc_impressions_prev"] >= 100).astype(int)

bins = [0, 3, 10, 20, 50, np.inf]
labels_ = ["top_3", "page_1", "striking", "page_3_5", "deep"]
prev["position_tier"] = pd.cut(prev["gsc_avg_position_prev"], bins=bins, labels=labels_)

feat = prev[prev["has_min_volume"] == 1].dropna(subset=["ctr_prev", "position_tier"]).copy()
feat["tier_avg_ctr_prev"] = feat.groupby("position_tier", observed=True)["ctr_prev"].transform("mean")

target_day["ctr_31"] = target_day["gsc_clicks"] / target_day["gsc_impressions"].replace(0, np.nan)
labeled = feat.merge(
    target_day[["client_hash_id", "content_hash_id", "ctr_31", "gsc_clicks"]],
    on=["client_hash_id", "content_hash_id"], how="inner"
).dropna(subset=["ctr_31"])

labeled["ctr_gap"] = labeled["tier_avg_ctr_prev"] - labeled["ctr_31"]
labeled["is_underperformer"] = (labeled["ctr_gap"] > 0).astype(int)

FEATURES = ["gsc_impressions_prev", "gsc_avg_position_prev", "position_tier", "ctr_prev", "has_min_volume"]
print(f"Five feature frame: {feat.shape[0]} rows, {len(FEATURES)} features")
print(feat[FEATURES].head())

X = pd.get_dummies(labeled[FEATURES], columns=["position_tier"])
y = labeled["is_underperformer"]

honest_model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X, y)
honest_score = honest_model.predict(X)
print(f"Honest Precision@50, 5 features: {precision_at_k(honest_score, y, 50):.3f}")

X_leaky = X.copy()
X_leaky["TRAP_target_day_clicks"] = labeled["gsc_clicks"].values
leaky_model = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_score = leaky_model.predict(X_leaky)
print(f"Leaky Precision@50, target day clicks added: {precision_at_k(leaky_score, y, 50):.3f}")

del X_leaky
print(f"Trap removed, kept honest number: Precision@50 = {precision_at_k(honest_score, y, 50):.3f}")

Five feature frame: 77540 rows, 5 features
    gsc_impressions_prev  gsc_avg_position_prev position_tier  ctr_prev  \
6                  201.0              41.208561      page_3_5  0.000000   
8                 2718.0               3.621756        page_1  0.001104   
9                12772.0               3.933929        page_1  0.001801   
10                 830.0               9.920193        page_1  0.000000   
11                2541.0               4.643188        page_1  0.003935   

    has_min_volume  
6                1  
8                1  
9                1  
10               1  
11               1  
Honest Precision@50, 5 features: 0.940
Leaky Precision@50, target day clicks added: 1.000
Trap removed, kept honest number: Precision@50 = 0.940


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


**Unbalanced panel + tier volume floors.** History depth differs sharply per client
(`dim_clients.gsc_data_start` ranges across the full ~17-month span), so a single
March-2026 window represents a very different fraction of history for a client with
17 months of data versus one just onboarded. Within March itself, `position_tier`
strata are not evenly sampled  `top_3` has a much lower median impression volume
than `page_1` (per the data dictionary: one click can move a low-volume tier's CTR
by ~1.9 percentage points), so `ctr_gap` for thin-volume tiers is noisier than for
high-volume ones even after the `has_min_volume` filter. This slice cannot tell us
whether a page's CTR gap is due to a bad title, a SERP feature (featured snippet,
People Also Ask) stealing clicks, or seasonal query intent only that a gap exists.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tier_volume = feat.groupby("position_tier", observed=True)["gsc_impressions_prev"].median().sort_values(ascending=False)
print("Median trailing impressions by position_tier, March, has_min_volume rows")
print(tier_volume)

client_history = con.sql(f"""
    SELECT gsc_data_start, COUNT(*) AS n_clients
    FROM read_parquet('{REL}/dim_clients.parquet')
    WHERE gsc_data_start IS NOT NULL
    GROUP BY gsc_data_start
    ORDER BY gsc_data_start
    LIMIT 5
""").df()
print("Earliest client gsc_data_start values, panel is unbalanced")
print(client_history)

Median trailing impressions by position_tier, March, has_min_volume rows
position_tier
top_3       896.0
page_1      619.0
page_3_5    565.0
striking    425.0
deep        183.5
Name: gsc_impressions_prev, dtype: float64
Earliest client gsc_data_start values, panel is unbalanced
  gsc_data_start  n_clients
0     2025-01-27          2
1     2025-02-11          1
2     2025-03-11          1
3     2025-06-07          1
4     2025-06-18          1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.